# Day 16 — Gold Streaming Mart: mart_charging_progress_live
**Source:** `silver/sl_vehicle_battery_live/` (Delta, streaming)  
**Sink:** `gold/mart_charging_progress_live/` (Delta, MERGE upsert)  
**Checkpoint:** `gold/_checkpoints/charging-progress-live/`  

### What this notebook builds
Reads the Silver streaming table and aggregates into **5-minute tumbling windows**  
per `vehicle_id + station_id`. Each window row captures:
- Latest / min / max / avg battery % in the window
- Average charging rate kW and battery temperature
- Overtemp flag (any reading > 45°C in this window = warning)
- Estimated minutes to full (minimum reading = closest to done)
- Event count (how many raw events fed this window)

### Why 5-minute windows?
In production, the Live Charging Power BI dashboard refreshes every 5 minutes.  
Each row in this mart is one tile on the dashboard — one vehicle at one station  
for one 5-minute period. The Cosmos DB `session_live` collection is also loaded  
from this mart for the mobile app (<2 sec read SLA).

### Merge key
`vehicle_id + station_id + window_start` — uniquely identifies a 5-min window.  
MERGE ensures a window that spans two Silver micro-batches gets updated, not duplicated.

**Run cells 1 to 7. Silver stream (Day 15) must be running first.**

In [ ]:
# Cell 1: Load secrets
SP_CLIENT_ID     = dbutils.secrets.get(scope='kv-ev-scope', key='sp-client-id')
SP_CLIENT_SECRET = dbutils.secrets.get(scope='kv-ev-scope', key='sp-client-secret')
SP_TENANT_ID     = dbutils.secrets.get(scope='kv-ev-scope', key='sp-tenant-id')
STORAGE_ACCOUNT  = dbutils.secrets.get(scope='kv-ev-scope', key='adls-account-name')
print('Secrets loaded.')

In [ ]:
# Cell 2: Configure ADLS Gen2 OAuth
spark.conf.set(f'fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net', 'OAuth')
spark.conf.set(f'fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net',
               'org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider')
spark.conf.set(f'fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net', SP_CLIENT_ID)
spark.conf.set(f'fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net', SP_CLIENT_SECRET)
spark.conf.set(f'fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net',
               f'https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token')
print(f'ADLS OAuth configured: {STORAGE_ACCOUNT}')

In [ ]:
# Cell 3: Define paths
SILVER_PATH     = f'abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/sl_vehicle_battery_live/'
GOLD_PATH       = f'abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/mart_charging_progress_live/'
CHECKPOINT_PATH = f'abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/_checkpoints/charging-progress-live/'

print(f'Silver:     {SILVER_PATH}')
print(f'Gold mart:  {GOLD_PATH}')
print(f'Checkpoint: {CHECKPOINT_PATH}')

In [ ]:
# Cell 4: Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_timestamp,
    window, max as fmax, min as fmin, avg as favg, count, when
)
from delta.tables import DeltaTable
print('Imports done.')

In [ ]:
# Cell 5: Gold transformation — foreachBatch function
#
# Aggregation grain: vehicle_id + station_id + 5-minute tumbling window
# Merge key:         vehicle_id + station_id + window_start
#
# A 5-minute window may span multiple Silver micro-batches (Silver triggers every 60s).
# MERGE ensures the window row is updated each time, never duplicated.
# Example: window 09:00-09:05 receives Silver batches at 09:01, 09:02, 09:03, 09:04, 09:05
#          — each batch updates the same Gold row, final state = complete window.

def build_gold_mart(batch_df, batch_id):
    row_count = batch_df.count()
    if row_count == 0:
        print(f'[Batch {batch_id}] Empty — skipping.')
        return

    # Parse event_ts string to timestamp for window() function
    parsed = batch_df.withColumn('event_ts_ts', to_timestamp(col('event_ts')))

    # 5-minute tumbling window aggregation
    agg_df = (
        parsed
        .groupBy(
            col('vehicle_id'),
            col('station_id'),
            window(col('event_ts_ts'), '5 minutes').alias('tw')
        )
        .agg(
            fmax('battery_pct').alias('max_battery_pct'),
            fmin('battery_pct').alias('min_battery_pct'),
            favg('battery_pct').alias('avg_battery_pct'),
            favg('charging_rate_kw').alias('avg_charging_rate_kw'),
            favg('battery_temp_c').alias('avg_battery_temp_c'),
            fmax('battery_temp_c').alias('max_battery_temp_c'),
            # overtemp_flag=1 if ANY reading in this window exceeded 45 C
            fmax(when(col('battery_temp_c') > 45.0, lit(1)).otherwise(lit(0))).alias('overtemp_flag'),
            # min estimated_minutes_to_full = vehicle closest to full charge
            fmin('estimated_minutes_to_full').alias('min_est_minutes_to_full'),
            fmax('session_id').alias('session_id'),
            fmax('charger_id').alias('charger_id'),
            count('event_id').alias('event_count')
        )
        .withColumn('window_start', col('tw.start'))
        .withColumn('window_end',   col('tw.end'))
        .drop('tw')
        .withColumn('_gold_updated_at', current_timestamp())
        .withColumn('_batch_id', lit(batch_id))
    )

    mart_count = agg_df.count()

    # MERGE into Gold mart on vehicle + station + window_start
    if DeltaTable.isDeltaTable(spark, GOLD_PATH):
        gold_table = DeltaTable.forPath(spark, GOLD_PATH)
        (
            gold_table.alias('gold')
            .merge(
                agg_df.alias('batch'),
                'gold.vehicle_id   = batch.vehicle_id '
                'AND gold.station_id   = batch.station_id '
                'AND gold.window_start = batch.window_start'
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        # First run — create Gold Delta table partitioned by vehicle_id
        (
            agg_df.write
            .format('delta').mode('overwrite')
            .partitionBy('vehicle_id')
            .save(GOLD_PATH)
        )

    print(f'[Batch {batch_id}] Silver rows in: {row_count} | 5-min windows upserted: {mart_count}')

print('build_gold_mart() defined.')

In [ ]:
# Cell 6: Start Gold streaming query
# Reads Silver Delta as a stream. trigger=120s — Gold aggregates every 2 minutes.
# Bronze: 30s -> Silver: 60s -> Gold: 120s  (each layer processes at its own cadence)

gold_query = (
    spark.readStream
    .format('delta')
    .option('ignoreChanges', 'true')
    .load(SILVER_PATH)
    .writeStream
    .foreachBatch(build_gold_mart)
    .trigger(processingTime='120 seconds')
    .option('checkpointLocation', CHECKPOINT_PATH)
    .start()
)

print(f'Gold stream started. ID: {gold_query.id}')
print(f'Trigger: 120s | Silver -> 5-min window aggregation -> MERGE -> Gold mart')
print(f'Gold mart: {GOLD_PATH}')

In [ ]:
# Cell 7: Monitor Gold stream (cancel anytime — stream keeps running)
import time
print('Gold stream live. Cancel to stop monitoring.\n')
while gold_query.isActive:
    p = gold_query.lastProgress
    if p:
        print(f'[Batch {p.get("batchId","?")}] '
              f'Input: {p.get("numInputRows",0):,} rows | '
              f'Rate: {p.get("processedRowsPerSecond",0.0):.1f} rows/sec')
    else:
        print('Waiting for first Gold batch...')
    time.sleep(120)
print('Gold stream stopped.')

In [ ]:
# Cell 8 (OPTIONAL): Verify Gold mart
# Run in a separate notebook or after cancelling Cell 7.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

gold_df = spark.read.format('delta').load(GOLD_PATH)
print(f'Total 5-min window rows in Gold mart: {gold_df.count():,}')

print('\nLatest window per vehicle (current charging state):')
latest_w = Window.partitionBy('vehicle_id').orderBy(col('window_start').desc())
(
    gold_df
    .withColumn('_rn', F.row_number().over(latest_w))
    .filter(col('_rn') == 1)
    .select('vehicle_id', 'station_id', 'window_start', 'window_end',
            'max_battery_pct', 'avg_charging_rate_kw', 'max_battery_temp_c',
            'overtemp_flag', 'min_est_minutes_to_full', 'event_count')
    .orderBy('vehicle_id')
    .show(truncate=False)
)

print('\nOvertemp alerts (overtemp_flag = 1, max_battery_temp_c > 45):')
alerts = gold_df.filter(col('overtemp_flag') == 1)
print(f'Overtemp windows: {alerts.count()}')
if alerts.count() > 0:
    alerts.select('vehicle_id', 'station_id', 'window_start',
                  'max_battery_temp_c', 'event_count').show(truncate=False)

print('\nDelta table history (last 5 operations):')
spark.sql(f"DESCRIBE HISTORY delta.`{GOLD_PATH}`").select(
    'version', 'timestamp', 'operation', 'operationMetrics'
).show(5, truncate=False)